In [1]:
import pandas as pd
df = pd.read_csv('stage3_output/stage3_replay_train.csv')
print(df.columns.tolist())
print(df['dataset_source'].value_counts())

['image_path', 'unified_label', 'fst_group', 'split', 'dataset_source', 'has_disease_label']
dataset_source
ISIC2019          13255
dermnet            8831
fitzpatrick17k     2553
PAD-UFES-20        1426
Name: count, dtype: int64


In [2]:
import torch
ckpt = torch.load('checkpoints/stage3_replay_best.pth', map_location='cpu')
print(ckpt.keys())

dict_keys(['model_state', 'classes', 'stage', 'best_composite'])


C:\Users\prakh\AppData\Local\Temp\ipykernel_23708\924688712.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('checkpoints/stage3_replay_best.pth', map_l

In [3]:
print(ckpt['classes'])
print(ckpt['model_state']['classifier.3.weight'].shape)

['MEL', 'BCC', 'SCC', 'AK', 'NEV', 'BKL', 'DF', 'VASC', 'SEK', 'TINEA', 'PSORIASIS', 'VITILIGO', 'MELASMA', 'FUNGAL', 'ECZEMA', 'URTICARIA']


KeyError: 'classifier.3.weight'

In [4]:
# Check your val CSV name and structure
import os
print(os.listdir('checkpoints/'))  # see all your checkpoint files

val_df = pd.read_csv('stage3_output/stage3_replay_val.csv')  # replace with actual name
print(val_df.columns.tolist())
print(val_df['dataset_source'].value_counts())
print(val_df['split'].value_counts())

['stage1.pth', 'stage1_finetuned.pth', 'stage2.pth', 'stage3.pth', 'stage3_replay_best.pth']
['image_path', 'unified_label', 'fst_group', 'split', 'dataset_source', 'has_disease_label']
dataset_source
ISIC2019          2494
dermnet           1559
fitzpatrick17k     352
PAD-UFES-20        324
Name: count, dtype: int64
split
val      3170
train    1559
Name: count, dtype: int64


In [5]:
# Cell 1 — verify checkpoint loads cleanly
import torch
import timm
import torch.nn as nn

ckpt = torch.load('checkpoints/stage3_replay_best.pth',
                   map_location='cpu',
                   weights_only=False)
print("Keys:", ckpt.keys())
print("Classes:", ckpt['classes'])
print("Stage:", ckpt['stage'])
print("Best composite:", ckpt['best_composite'])

c:\Users\prakh\anaconda3\envs\dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Keys: dict_keys(['model_state', 'classes', 'stage', 'best_composite'])
Classes: ['MEL', 'BCC', 'SCC', 'AK', 'NEV', 'BKL', 'DF', 'VASC', 'SEK', 'TINEA', 'PSORIASIS', 'VITILIGO', 'MELASMA', 'FUNGAL', 'ECZEMA', 'URTICARIA']
Stage: 3_replay
Best composite: 0.36392580046633954


In [6]:
# Cell 2 — find exact classifier layer names
for k, v in ckpt['model_state'].items():
    if any(x in k for x in ['classifier','head','fc']):
        print(k, v.shape)

backbone.conv_head.weight torch.Size([1408, 352, 1, 1])
head.2.weight torch.Size([1408])
head.2.bias torch.Size([1408])
head.2.running_mean torch.Size([1408])
head.2.running_var torch.Size([1408])
head.2.num_batches_tracked torch.Size([])
head.4.weight torch.Size([16, 1408])
head.4.bias torch.Size([16])


In [7]:
# See what keys the checkpoint expects
for k in ckpt['model_state'].keys():
    print(k)

backbone.conv_stem.weight
backbone.bn1.weight
backbone.bn1.bias
backbone.bn1.running_mean
backbone.bn1.running_var
backbone.bn1.num_batches_tracked
backbone.blocks.0.0.conv_dw.weight
backbone.blocks.0.0.bn1.weight
backbone.blocks.0.0.bn1.bias
backbone.blocks.0.0.bn1.running_mean
backbone.blocks.0.0.bn1.running_var
backbone.blocks.0.0.bn1.num_batches_tracked
backbone.blocks.0.0.se.conv_reduce.weight
backbone.blocks.0.0.se.conv_reduce.bias
backbone.blocks.0.0.se.conv_expand.weight
backbone.blocks.0.0.se.conv_expand.bias
backbone.blocks.0.0.conv_pw.weight
backbone.blocks.0.0.bn2.weight
backbone.blocks.0.0.bn2.bias
backbone.blocks.0.0.bn2.running_mean
backbone.blocks.0.0.bn2.running_var
backbone.blocks.0.0.bn2.num_batches_tracked
backbone.blocks.0.1.conv_dw.weight
backbone.blocks.0.1.bn1.weight
backbone.blocks.0.1.bn1.bias
backbone.blocks.0.1.bn1.running_mean
backbone.blocks.0.1.bn1.running_var
backbone.blocks.0.1.bn1.num_batches_tracked
backbone.blocks.0.1.se.conv_reduce.weight
backbone.b

In [10]:
import torch
import torch.nn as nn
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
UNIFIED_CLASSES = ['MEL','BCC','SCC','AK','NEV','BKL','DF','VASC','SEK','TINEA','PSORIASIS','VITILIGO','MELASMA','FUNGAL','ECZEMA','URTICARIA']

class EfficientNetB2Unified(nn.Module):
    def __init__(self, num_classes=16):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=False, num_classes=0, global_pool='')
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(1408),
            nn.Dropout(0.3),
            nn.Linear(1408, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone.forward_features(x))

model = EfficientNetB2Unified(num_classes=len(UNIFIED_CLASSES)).to(DEVICE)

ckpt = torch.load(r"checkpoints\stage3_replay_best.pth", map_location=DEVICE, weights_only=True)
state = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
model.load_state_dict(state, strict=True)
model.eval()

print("Loaded:", r"checkpoints\stage3_replay_best.pth")

Loaded: checkpoints\stage3_replay_best.pth


In [11]:
for k in model.state_dict().keys():
    print(k)

backbone.conv_stem.weight
backbone.bn1.weight
backbone.bn1.bias
backbone.bn1.running_mean
backbone.bn1.running_var
backbone.bn1.num_batches_tracked
backbone.blocks.0.0.conv_dw.weight
backbone.blocks.0.0.bn1.weight
backbone.blocks.0.0.bn1.bias
backbone.blocks.0.0.bn1.running_mean
backbone.blocks.0.0.bn1.running_var
backbone.blocks.0.0.bn1.num_batches_tracked
backbone.blocks.0.0.se.conv_reduce.weight
backbone.blocks.0.0.se.conv_reduce.bias
backbone.blocks.0.0.se.conv_expand.weight
backbone.blocks.0.0.se.conv_expand.bias
backbone.blocks.0.0.conv_pw.weight
backbone.blocks.0.0.bn2.weight
backbone.blocks.0.0.bn2.bias
backbone.blocks.0.0.bn2.running_mean
backbone.blocks.0.0.bn2.running_var
backbone.blocks.0.0.bn2.num_batches_tracked
backbone.blocks.0.1.conv_dw.weight
backbone.blocks.0.1.bn1.weight
backbone.blocks.0.1.bn1.bias
backbone.blocks.0.1.bn1.running_mean
backbone.blocks.0.1.bn1.running_var
backbone.blocks.0.1.bn1.num_batches_tracked
backbone.blocks.0.1.se.conv_reduce.weight
backbone.b

In [2]:
# DDI metadata
import pandas as pd
ddi = pd.read_csv('data/ddi_metadata.csv')  # adjust filename
print(ddi.columns.tolist())
print(ddi.head(3))
print(ddi.dtypes)

['Unnamed: 0', 'DDI_ID', 'DDI_file', 'skin_tone', 'malignant', 'disease']
   Unnamed: 0  DDI_ID    DDI_file  skin_tone  malignant            disease
0           0       1  000001.png         56       True   melanoma-in-situ
1           1       2  000002.png         56       True   melanoma-in-situ
2           2       3  000003.png         56       True  mycosis-fungoides
Unnamed: 0     int64
DDI_ID         int64
DDI_file      object
skin_tone      int64
malignant       bool
disease       object
dtype: object


In [4]:
# DDI-2 metadata
ddi2 = pd.read_csv('data/ddi2diversedermatologyimages2/final_DDI2_Asian_spreadsheet.csv')  # adjust filename
print(ddi2.columns.tolist())
print(ddi2.head(3))
print(ddi2.dtypes)

['deidentified_patient_id', 'more_than_1_photo_for_this_patient', 'photo_id', 'fitzpatrick_skin_type', 'anatomical_site_detailed', 'anatomical_site_general', 'diagnosis_detailed', 'diagnosis_general', 'dermatological_presentation', 'benign/malignant', 'common/uncommon', 'cropped?', 'extra_markings', 'sex', 'race', 'ethnicity', 'self-described_ethnic_background', 'language', 'interpreter_needed']
   deidentified_patient_id more_than_1_photo_for_this_patient  \
0                        1                                  N   
1                        2                                  N   
2                        4                                  N   

            photo_id fitzpatrick_skin_type anatomical_site_detailed  \
0  502948221_cropped                     3             Right cheek    
1            8579589                     3            Left ear lobe   
2          103634700                  3, 4                  Scrotum   

  anatomical_site_general             diagnosis_detaile